# Scientific ratios — dimensionless groups (Buckingham Pi)

[`dgs/dimensional_analysis.py`](../dgs/dimensional_analysis.py) already checks
that both sides of an equation reduce to the *same* units (Coulomb's law,
Gauss's law, the EM wave speed, ...). That answers "is this equation
consistent?" A different and in some ways more powerful question is: **which
combinations of quantities have NO units at all?**

A dimensionless ratio is unit-system-independent by construction — the same
number whether you measured in meters or furlongs — and that's exactly why a
single number like the Reynolds number or the fine structure constant can
characterize an entire physical regime. This notebook adds that class of
check to the module, then applies it to something specific to this lab: the
dispersion parameter `D` in `gs_core.py` already has a hard-coded regime
threshold (`|D| >= 100` warns, `|D| >= 5000` is "needed for convergence") —
we derive *why*, as a genuine dimensionless ratio, and verify it against the
algorithm's actual behavior.


In [1]:
import sys, pathlib
import numpy as np
import scipy.constants as sc
from sympy.physics import units as u

REPO = pathlib.Path(r"D:/Summer2026/Dispersion-Assisted-GS-Phase-Recovery")
sys.path.insert(0, str(REPO))
from dgs import dimensional_analysis as da
from dgs import gs_core as gs

checks = []


def check(label, condition):
    checks.append((label, bool(condition)))
    print(f"{'PASS' if condition else 'FAIL'}  —  {label}")


## 1. Reynolds number — the textbook Buckingham Pi group

$$\mathrm{Re} = \frac{\rho v L}{\mu}$$

Four quantities, each with its own units (density, velocity, length,
viscosity) — combine into a pure number. That number, not the individual
quantities, is what tells you whether a flow is laminar or turbulent.


In [2]:
re_expr = (u.kilogram / u.meter**3) * (u.meter / u.second) * u.meter / (u.pascal * u.second)
print("Re = rho*v*L/mu  is dimensionless:", da.is_dimensionless(re_expr))
check("Reynolds number combination is dimensionless", da.check_reynolds_number())

# Drop the viscosity term: rho*v*L is a mass flow rate per unit width, NOT
# dimensionless -- confirms the checker actually distinguishes the two.
rho_v_L = (u.kilogram / u.meter**3) * (u.meter / u.second) * u.meter
print("rho*v*L (no /mu) is dimensionless:", da.is_dimensionless(rho_v_L))
check("Dropping viscosity correctly breaks dimensionlessness", not da.is_dimensionless(rho_v_L))


Re = rho*v*L/mu  is dimensionless: True
PASS  —  Reynolds number combination is dimensionless
rho*v*L (no /mu) is dimensionless: False
PASS  —  Dropping viscosity correctly breaks dimensionlessness


## 2. The fine structure constant — independently cross-checked

$$\alpha = \frac{e^2}{4\pi\varepsilon_0\hbar c}$$

Charge squared, over permittivity times action times velocity — four SI
constants, each dimensioned, combining to a pure number ≈ 1/137. We compute
it from SymPy's own unit system, then check it against `scipy.constants`, an
entirely separate constants table, as an honest independent cross-check
(not the same computation done twice).


In [3]:
alpha, alpha_is_dimensionless = da.fine_structure_constant()
print(f"alpha (SymPy units)      = {alpha:.10f}   (1/alpha = {1/alpha:.4f})")
print(f"alpha (scipy.constants)  = {sc.fine_structure:.10f}   (1/alpha = {1/sc.fine_structure:.4f})")

check("alpha is dimensionless", alpha_is_dimensionless)
check("alpha matches scipy.constants.fine_structure (independent source)",
      abs(alpha - sc.fine_structure) < 1e-6)


alpha (SymPy units)      = 0.0072973526   (1/alpha = 137.0360)
alpha (scipy.constants)  = 0.0072973526   (1/alpha = 137.0360)
PASS  —  alpha is dimensionless
PASS  —  alpha matches scipy.constants.fine_structure (independent source)


## 3. The lab's own ratio: when does dispersion give you a real-time Fourier transform?

`gs_core.py`'s `H(\nu) = \exp(i\pi D\nu^2)` sweeps a quadratic phase across
the normalized frequency axis $\nu\in[-0.5,0.5)$. At the Nyquist edge
($\nu=0.5$) that phase is

$$\theta_{\max} = \pi D (0.5)^2 = \frac{\pi D}{4}.$$

The stationary-phase argument from
[`griffiths_1_42_1_43_cylindrical_unit_vectors_divergence_theorem.ipynb`](../../griffiths_1_42_1_43_cylindrical_unit_vectors_divergence_theorem.ipynb)
(the sifting → real-time-Fourier-transform bridge) only holds when the phase
sweeps through **many radians** across the usable band — otherwise there's no
sharp stationary point to sift out, just a slowly-varying phase that barely
moves. `|D|/4` (in units of $\pi$ radians) is exactly that dimensionless
sweep — and it's exactly the number `gs_core._check_dispersion` is gating on
when it warns below `|D|=100` and documents `|D|>=5000` as needed.


In [4]:
D_used = 5000     # the default make_qpsk_measurements/make_measurements dispersion
D_small = 10      # deliberately below gs_core's own warn threshold of 100

ratio_used = da.dispersion_regime_parameter(-D_used)
ratio_small = da.dispersion_regime_parameter(-D_small)

print(f"|D|/4 at D={-D_used} (gs_core default):        {ratio_used:.1f}   (>>1: deep in the real-time-FT regime)")
print(f"|D|/4 at D={-D_small} (below gs_core's warn cutoff): {ratio_small:.2f}   (~1: near-field, no clean stationary point)")

check("Regime parameter at the lab's default D is >> 1", ratio_used > 100)
check("Regime parameter at small D is only order-1", ratio_small < 10)
check("Regime parameter grows with |D| (matches gs_core's own threshold logic)", ratio_used > ratio_small)


|D|/4 at D=-5000 (gs_core default):        1250.0   (>>1: deep in the real-time-FT regime)
|D|/4 at D=-10 (below gs_core's warn cutoff): 2.50   (~1: near-field, no clean stationary point)
PASS  —  Regime parameter at the lab's default D is >> 1
PASS  —  Regime parameter at small D is only order-1
PASS  —  Regime parameter grows with |D| (matches gs_core's own threshold logic)


### Does the ratio actually predict GS convergence?

A dimensionless number is only useful if it predicts something real. Run the
*actual* Gerchberg–Saxton phase retrieval at both dispersions and check that
the large-ratio case converges to low phase error while the small-ratio case
does not — the empirical behavior `gs_core`'s own docstrings describe.


In [5]:
data_large_D = gs.make_qpsk_measurements(n_symbols=64, sps=8, D1=-D_used, D2=-5750.0, snr_db=30.0)
phi_est_large, errors_large = gs.retrieve_phase(
    data_large_D["I1"], data_large_D["I2"], data_large_D["D1"], data_large_D["D2"], n_iter=80
)

data_small_D = gs.make_qpsk_measurements(n_symbols=64, sps=8, D1=-D_small, D2=-11.5, snr_db=30.0)
phi_est_small, errors_small = gs.retrieve_phase(
    data_small_D["I1"], data_small_D["I2"], data_small_D["D1"], data_small_D["D2"], n_iter=80
)


def rms_phase_error(phi_true, phi_est):
    offset = np.angle(np.mean(np.exp(1j * (phi_true - phi_est))))
    delta = np.angle(np.exp(1j * (phi_est - phi_true + offset)))
    return float(np.sqrt(np.mean(delta**2)))


rms_large = rms_phase_error(data_large_D["phi_true"], phi_est_large)
rms_small = rms_phase_error(data_small_D["phi_true"], phi_est_small)

print(f"|D|/4 = {ratio_used:7.1f}  ->  RMS phase error = {rms_large:.4f} rad ({np.degrees(rms_large):.1f} deg)")
print(f"|D|/4 = {ratio_small:7.1f}  ->  RMS phase error = {rms_small:.4f} rad ({np.degrees(rms_small):.1f} deg)")

check("Large regime-parameter case converges to a small phase error", rms_large < 0.2)
check("Small regime-parameter case is measurably worse than the large one", rms_small > rms_large)


|D|/4 =  1250.0  ->  RMS phase error = 0.0394 rad (2.3 deg)
|D|/4 =     2.5  ->  RMS phase error = 1.0488 rad (60.1 deg)
PASS  —  Large regime-parameter case converges to a small phase error
PASS  —  Small regime-parameter case is measurably worse than the large one


C:\Users\mrjel\AppData\Local\Temp\ipykernel_4624\442527431.py:7: UserWarning: |D1|=10.0 < 100. GS convergence requires |D| ≥ 5000 (normalized). Physical: -695 ps/nm → D_norm ≈ -5000. Current value will likely stagnate.
  phi_est_small, errors_small = gs.retrieve_phase(
C:\Users\mrjel\AppData\Local\Temp\ipykernel_4624\442527431.py:7: UserWarning: |D2|=11.5 < 100. GS convergence requires |D| ≥ 5000 (normalized). Physical: -695 ps/nm → D_norm ≈ -5000. Current value will likely stagnate.
  phi_est_small, errors_small = gs.retrieve_phase(


The dimensionless ratio isn't just a bookkeeping label — it predicts
which of the two runs above actually recovers the phase. That's the point of
a Buckingham Pi group: one unit-free number standing in for the whole
question of "which physical regime am I in?"

## Final grade

In [6]:
failures = [label for label, ok in checks if not ok]
print(f"{len(checks) - len(failures)}/{len(checks)} checks passed")

if failures:
    raise AssertionError("Failed checks: " + ", ".join(failures))
else:
    print("\nALL CHECKS PASSED — Reynolds number, the fine structure constant, and "
          "gs_core's own dispersion threshold all verified as genuine dimensionless ratios.")


9/9 checks passed

ALL CHECKS PASSED — Reynolds number, the fine structure constant, and gs_core's own dispersion threshold all verified as genuine dimensionless ratios.
